# Heaps & Priority Queues in Python — Implementation & Algorithmic Patterns

> **Topic:** Heaps & Priority Queues | **Folder:** Data Structures & Algorithms

A **Heap** is a specialized tree-based data structure that satisfies the **Heap Property**:
- **Min-Heap**: The parent node is always smaller than or equal to its children ($Key(Parent) \le Key(Child)$).
- **Max-Heap**: The parent node is always greater than or equal to its children ($Key(Parent) \ge Key(Child)$).

Heaps are stored efficiently in **1D arrays** without pointer overhead.

---

## Table of Contents
1. [Heap Concepts & Array Indexing Formulas](#1.-Heap-Concepts-&-Array-Indexing-Formulas)
2. [The Built-in `heapq` Module](#2.-The-Built-in-`heapq`-Module)
3. [Building a Max-Heap in Python](#3.-Building-a-Max-Heap-in-Python)
4. [Building a Min-Heap from Scratch (`sift_up` & `sift_down`)](#4.-Building-a-Min-Heap-from-Scratch-(sift_up-&-sift_down))
5. [Algorithmic Pattern 1: Top K Frequent / Largest Elements](#5.-Algorithmic-Pattern-1:-Top-K-Frequent-/-Largest-Elements)
6. [Algorithmic Pattern 2: Merge K Sorted Lists](#6.-Algorithmic-Pattern-2:-Merge-K-Sorted-Lists)
7. [Algorithmic Pattern 3: Find Median from Data Stream (Dual Heap)](#7.-Algorithmic-Pattern-3:-Find-Median-from-Data-Stream-(Dual-Heap))
8. [Quick Reference Card](#8.-Quick-Reference-Card)


---
## 1. Heap Concepts & Array Indexing Formulas

For any node at array index $i$ (0-indexed):
- **Parent**: `(i - 1) // 2`
- **Left Child**: `2 * i + 1`
- **Right Child**: `2 * i + 2`

| Heap Operation | Time Complexity | Space Complexity |
|----------------|-----------------|------------------|
| **`heapify(list)`** | $O(n)$ Linear Time! | $O(1)$ in-place |
| **`heappush(heap, val)`** | $O(\log n)$ | $O(1)$ |
| **`heappop(heap)`** | $O(\log n)$ | $O(1)$ |
| **`peek(heap)`** | $O(1)$ | $O(1)$ |


---
## 2. The Built-in `heapq` Module

Python's `heapq` module implements a **Min-Heap** on standard lists.


In [ ]:
import heapq

# 1. Convert list to min-heap in O(n) time
nums = [5, 1, 9, 3, 7, 4]
heapq.heapify(nums)
print("Heapified list (min at index 0):", nums)

# 2. Push and Pop
heapq.heappush(nums, 2)
print("After pushing 2: min is", nums[0])
print("Popped minimum:", heapq.heappop(nums))
print("Popped minimum:", heapq.heappop(nums))


---
## 3. Building a Max-Heap in Python

To simulate a **Max-Heap** using `heapq`, multiply numerical values by `-1` upon insertion!


In [ ]:
# Simulating Max-Heap using negation
max_heap = []
values = [10, 50, 20, 40, 30]

for val in values:
    heapq.heappush(max_heap, -val)  # Push negative

print("Largest element:", -heapq.heappop(max_heap))  # Returns 50
print("Second largest:", -heapq.heappop(max_heap))   # Returns 40


---
## 4. Building a Min-Heap from Scratch (`sift_up` & `sift_down`)


In [ ]:
class MinHeap:
    def __init__(self):
        self.heap = []

    def push(self, val):
        self.heap.append(val)
        self._sift_up(len(self.heap) - 1)

    def pop(self):
        if not self.heap: raise IndexError("pop from empty heap")
        min_val = self.heap[0]
        last_val = self.heap.pop()
        if self.heap:
            self.heap[0] = last_val
            self._sift_down(0)
        return min_val

    def _sift_up(self, i):
        parent = (i - 1) // 2
        while i > 0 and self.heap[i] < self.heap[parent]:
            self.heap[i], self.heap[parent] = self.heap[parent], self.heap[i]
            i = parent
            parent = (i - 1) // 2

    def _sift_down(self, i):
        n = len(self.heap)
        while True:
            left = 2 * i + 1
            right = 2 * i + 2
            smallest = i
            if left < n and self.heap[left] < self.heap[smallest]: smallest = left
            if right < n and self.heap[right] < self.heap[smallest]: smallest = right
            if smallest == i: break
            self.heap[i], self.heap[smallest] = self.heap[smallest], self.heap[i]
            i = smallest

# Testing MinHeap
mh = MinHeap()
for x in [15, 10, 20, 5, 30]: mh.push(x)
print("Popped from MinHeap:", [mh.pop() for _ in range(5)])


---
## 5. Algorithmic Pattern 1: Top K Largest Elements

Maintains a min-heap of size $k$. Running time: **$O(n \log k)$** instead of $O(n \log n)$ sorting.


In [ ]:
def find_top_k_largest(nums, k):
    min_heap = nums[:k]
    heapq.heapify(min_heap)
    for x in nums[k:]:
        if x > min_heap[0]:
            heapq.heapreplace(min_heap, x)
    return sorted(min_heap, reverse=True)

data = [3, 2, 1, 5, 6, 4]
print("Top 2 largest elements:", find_top_k_largest(data, 2))


---
## 6. Algorithmic Pattern 2: Merge K Sorted Lists


In [ ]:
def merge_k_sorted_lists(lists):
    min_heap = []
    # Store (value, list_index, element_index)
    for i, l in enumerate(lists):
        if l:
            heapq.heappush(min_heap, (l[0], i, 0))
            
    result = []
    while min_heap:
        val, l_idx, e_idx = heapq.heappop(min_heap)
        result.append(val)
        if e_idx + 1 < len(lists[l_idx]):
            heapq.heappush(min_heap, (lists[l_idx][e_idx + 1], l_idx, e_idx + 1))
            
    return result

sorted_lists = [[1, 4, 7], [2, 5, 8], [3, 6, 9]]
print("Merged K Lists:", merge_k_sorted_lists(sorted_lists))


---
## 7. Algorithmic Pattern 3: Find Median from Data Stream (Dual Heap)


In [ ]:
class MedianFinder:
    def __init__(self):
        self.small = []  # Max-Heap (stores smaller half)
        self.large = []  # Min-Heap (stores larger half)

    def addNum(self, num: int) -> None:
        heapq.heappush(self.small, -num)
        # Ensure all small <= all large
        if self.small and self.large and (-self.small[0] > self.large[0]):
            val = -heapq.heappop(self.small)
            heapq.heappush(self.large, val)
        # Balance size
        if len(self.small) > len(self.large) + 1:
            val = -heapq.heappop(self.small)
            heapq.heappush(self.large, val)
        elif len(self.large) > len(self.small):
            val = heapq.heappop(self.large)
            heapq.heappush(self.small, -val)

    def findMedian(self) -> float:
        if len(self.small) > len(self.large):
            return -self.small[0]
        return (-self.small[0] + self.large[0]) / 2.0

mf = MedianFinder()
mf.addNum(1); mf.addNum(2)
print("Median after [1, 2]:", mf.findMedian())   # 1.5
mf.addNum(3)
print("Median after [1, 2, 3]:", mf.findMedian()) # 2.0


---
## 8. Quick Reference Card


In [ ]:
# ==================================================================
# HEAPS – QUICK REFERENCE
# ==================================================================
import heapq

h = [5, 1, 3]
heapq.heapify(h)                 # O(n) min-heapify
heapq.heappush(h, 2)             # O(log n) push
min_val = heapq.heappop(h)       # O(log n) pop min
top_3 = heapq.nlargest(3, [1, 9, 5, 2, 8])  # nlargest
print("Top 3:", top_3)


---
## Summary

| Operation / Pattern | Time Complexity | Purpose |
|---------------------|-----------------|---------|
| **`heapify`** | $O(n)$ | Fast heap construction |
| **`heappush` / `heappop`** | $O(\log n)$ | Priority Queue insertions & deletions |
| **Top K Elements** | $O(n \log k)$ | Find K largest/smallest elements |
| **Merge K Lists** | $O(N \log k)$ | Multi-way stream sorting |
| **Dual Heap Median** | $O(\log n)$ insert, $O(1)$ median | Dynamic stream median tracking |

---
*Next up: **Trees & Binary Search Trees***
